In [ ]:
%%time
import sys
print(sys.version)
print(sys.executable)
%pip install rasterio numpy pandas geopandas scipy dask[distributed] bokeh rioxarray xarray pathlib openpyxl


In [ ]:
%%time
import pandas as pd
import os
from collections import defaultdict, Counter
import datetime
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import rasterio.mask
import shutil  # This module allows for file copying
from rasterio.warp import reproject, Resampling
from pathlib import Path

# Set the base directory 
base_dir = Path("F:\Runpod\Master_Thesis_EQ47")  # Adjusted the Base Directory
num_workers = 12  # Adjusted the number of workers for parallel processing
threads_per_worker = 2
memory_limit = '5GB'

# Define paths for input and output directories
EPSG_target = 32647

In [ ]:
%%time
from pathlib import Path
#Range of date interesting for the analysis
date_start = datetime.date(2021, 5, 23)
date_end = datetime.date(2021, 8, 21)
threshold_limit = 0
# Input and Output paths for Pre image processing
Input_dir_Hotspot = base_dir / "Datasource/EQ/Hotspot/input/fire_archive_M-C61_635378.csv"
Output_dir_Hotspot = base_dir / "Datasource/EQ/Hotspot/output/Hotspot"
Output_dir_Hotspotcluster = base_dir / "Datasource/EQ/Hotspot/output/HotspotCluster"
output_folder_SHP = base_dir / "Datasource/EQ/Hotspot/output/HotspotCluster_SHP"
reference_raster_path = base_dir / "Datasource/EQ/Hotspot/input/MOD11A1_LST_Day_1km_2010_142.tif"
PTR_output = base_dir / "Datasource/EQ/Hotspot/output/PointtoRaster"
FB_output = base_dir / "Datasource/EQ/Hotspot/output/MaskingLayer"
reportpath = base_dir / "Report"

#Input and Output paths for Thermal Anomaly
Input_Raw_MODIS = base_dir / "Datasource/MODIStsp/MODIStsp_Celsius"
TempwithoutError = base_dir / "Datasource/MODIStsp/TempwithoutError"
DeltaT_LC = base_dir / "Datasource/MODIStsp/DeltaTempLC"
RETIRA_LC = base_dir / "Datasource/TA/OriginalTA_LC"
RETIRA_Majority_LC = base_dir / "Datasource/TA/OriginalTA_Majority_LC"
RETIRA_Continue_LC = base_dir / "Datasource/TA/OriginalTA_Continue_LC"
LC_Folder= base_dir / "Datasource/MODIStsp/LU/LandCover_Type_Yearly_500m_v61/LC1"
outputLC_Folder = base_dir / "Datasource/Landcover/LC"

#AOI Clip with Shapefile
AOI_shp = base_dir / "Datasource/Master_Thesis_AOI/AOI_Thesis.shp"
RETIRA_LC_Clip_Final = base_dir / "Datasource/TA/OriginalTA_LC_Clip_Final"
RETIRA_Majority_LC_Clip_Final = base_dir / "Datasource/TA/OriginalTA_Majority_LC_Clip_Final"
RETIRA_Continue_LC_Clip_Final = base_dir / "Datasource/TA/OriginalTA_Continue_LC_Clip_Final"

In [ ]:
%%time
input_OD = base_dir / "Datasource/MODIStsp/MOD/Surf_Temp_Daily_1Km_v61/LST_Day_1km"
input_ON = base_dir / "Datasource/MODIStsp/MOD/Surf_Temp_Daily_1Km_v61/LST_Night_1km"
input_YD = base_dir / "Datasource/MODIStsp/MYD/Surf_Temp_Daily_1Km_v61/LST_Day_1km"
input_YN = base_dir / "Datasource/MODIStsp/MYD/Surf_Temp_Daily_1Km_v61/LST_Night_1km"

K TO C

In [ ]:
%%time
import os
import numpy as np
import rasterio
import geopandas as gpd  # using GeoPandas for reading shapefiles
from rasterio.mask import mask
import gc
from dask.distributed import Client, LocalCluster
from pathlib import Path
from shapely.geometry import mapping # Using for converting geometries to GeoJSON format for Dask compatibility

# ───── [1] DASK SETUP ───────────────────────────────────────────────
try:
    client.close()
    cluster.close()
except:
    pass

cluster = LocalCluster(n_workers=num_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit)
client = Client(cluster)

# ───── [2] AOI PREPARATION (Reading Shapefile) ──────────────────────────
gdf_aoi = None
aoi_shapes = None # This is the variable that will be sent to the Worker

if 'AOI_shp' in locals() or 'AOI_shp' in globals():
    aoi_path = Path(AOI_shp)
    ref_path = Path(reference_raster_path)
    
    if aoi_path.exists() and ref_path.exists():
        try:
            # 1. Read AOI Shapefile
            gdf_aoi = gpd.read_file(aoi_path)
            
            # 2. Read CRS from Reference Raster (LST 1km)
            with rasterio.open(ref_path) as ref_src:
                ref_crs = ref_src.crs
            
            # 3. Inspect and Reproject AOI to match Reference CRS if necessary
            if gdf_aoi.crs != ref_crs:
                print(f"🔄 CRS Mismatch: Changing {gdf_aoi.crs} ⮕ {ref_crs}")
                gdf_aoi = gdf_aoi.to_crs(ref_crs)
            else:
                print(f"✅ CRS Match: AOI CRS and Reference CRS match ({ref_crs})")
            
            # 4. Convert GeoDataFrame to List of Geometries (GeoJSON format)
            # Fix the 'str' object has no attribute 'get' issue and facilitate Dask compatibility
            aoi_shapes = [mapping(g) for g in gdf_aoi.geometry]
            
            print(f"📌 AOI Ready: {aoi_path.name} ({len(aoi_shapes)} shapes)")
            
        except Exception as e:
            print(f"⚠️ Found Error in AOI: {e}")
            gdf_aoi = None
            aoi_shapes = None
    else:
        # กรณีหาไฟล์ไม่เจอ ให้แจ้งพาธที่ระบบพยายามหา
        if not aoi_path.exists(): print(f"ℹ️ Not found AOI at: {aoi_path}")
        if not ref_path.exists(): print(f"ℹ️ Not found Reference at: {ref_path}")
else:
    print("ℹ️ No AOI_shp variable declared in the system")

# ───── [3] WORKER FUNCTION ──────────────────────────────────────────

def process_kelvin_to_celsius_worker(file_info, output_folder, shapes=None):
    input_path, file_name = file_info
    out_path = os.path.join(output_folder, file_name)
    
    try:
        with rasterio.open(input_path) as src:
            # Inscpect the CRS of the source raster
            
            
            if shapes is not None:
                # 1. Clip Images
                out_image, out_transform = mask(src, shapes, crop=True)
                data = out_image[0].astype(np.float32)
                meta = src.meta.copy()
                meta.update({
                    "height": data.shape[0],
                    "width": data.shape[1],
                    "transform": out_transform
                })
            else:
                # 1. Read Image Normally (No Clipping)
                data = src.read(1).astype(np.float32)
                meta = src.meta.copy()

            # 2. Convert Kelvin to Celsius
            nodata = src.nodata
            valid_mask = (data != nodata)
            data[valid_mask] = data[valid_mask] - 273.15
            
            # 3. Update Meta for Writing File
            meta.update({
                'dtype': 'float32',
                'compress': 'lzw'
            })

            with rasterio.open(out_path, 'w', **meta) as dst:
                dst.write(data, 1)
                
        mode = "Clipped" if shapes else "Full-Extent"
        return f"✔ {mode}: {file_name}"
    
    except Exception as e:
        return f"❌ Error {file_name}: {str(e)}"

# ───── [4] MAIN EXECUTION ───────────────────────────────────────────
input_output_pairs = [
    (input_OD, Input_Raw_MODIS),
    (input_ON, Input_Raw_MODIS),
    (input_YD, Input_Raw_MODIS),
    (input_YN, Input_Raw_MODIS),
]

print(f"🚀 Starting processing with {num_workers} Workers...")

for idx, (input_dir, output_dir) in enumerate(input_output_pairs, 1):
    if not os.path.exists(input_dir):
        continue
        
    print(f"\n--- Set {idx}: {os.path.basename(input_dir)} ---")
    os.makedirs(output_dir, exist_ok=True)
    
    tif_files = [(os.path.join(input_dir, f), f) for f in os.listdir(input_dir) if f.lower().endswith('.tif')]
    
    if not tif_files:
        continue
    
    # Set Job to Dask
    futures = client.map(process_kelvin_to_celsius_worker, tif_files, 
                         output_folder=output_dir, 
                         shapes=aoi_shapes)
    
    results = client.gather(futures)
    gc.collect()
    print(f"✨ Completed {len(results)} files")

print(f"\n✅ All processing completed!")

# Close Dask
client.close()
cluster.close()

EXCEL TO Excel Per DATE

In [ ]:
%%time
import os
import pandas as pd
from dask.distributed import Client, LocalCluster
from pathlib import Path

# === 1. Setup Dask Cluster ===
if __name__ == "__main__":
    # Setting the number of workers, threads per worker and momory limit for each worker
    cluster = LocalCluster(
        n_workers=num_workers,           # equivalent to max_workers=8
        threads_per_worker=threads_per_worker,  
        memory_limit=memory_limit     # Limit RAM per worker to avoid exceeding system memory
    )
    client = Client(cluster)
    print(f"🚀 Dashboard: {client.dashboard_link}")

    # === 2. Config Paths ===
    input_file = Input_dir_Hotspot
    output_dir = Output_dir_Hotspotcluster
    os.makedirs(output_dir, exist_ok=True)

    # === 3. Worker Function (Pandas ล้วนๆ) ===
    def pandas_worker(df_group, group_name, out_path):
        """Receive Pandas DataFrame to write to Excel"""
        try:
            date_obj, sat, dn = group_name
            filename = f"{date_obj.strftime('%d%m%Y')}_{sat}_{dn}.xlsx"
            file_path = os.path.join(out_path, filename)
            
            # ใช้ Pandas write ปกติ
            df_group.to_excel(file_path, index=False, engine='openpyxl')
            return f"Success: {filename}"
        except Exception as e:
            return f"Error: {str(e)}"

    # === 4. Main Process ===
    print("📑 Loading data with Pandas...")
    df = pd.read_csv(input_file)
    df['acq_date'] = pd.to_datetime(df['acq_date'], dayfirst=True)

    print("📦 Grouping...")
    grouped = df.groupby(['acq_date', 'satellite', 'daynight'])

    print("🔥 Submitting tasks to Dask Distributed...")
    futures = []
    for name, group in grouped:
        future = client.submit(pandas_worker, group, name, str(output_dir))
        futures.append(future)

    # === 5. Gather Results ===
    print(f"⏳ Waiting for {len(futures)} tasks to finish...")
    results = client.gather(futures) # คล้ายๆ list(executor.map)

    print(f"✅ Done! Created {len(results)} files.")
    
    client.close()
    cluster.close()

Excel to SHP

In [ ]:
%%time
import os
import datetime
import pandas as pd
import geopandas as gpd
from collections import Counter
import dask
from dask.distributed import Client, LocalCluster

# === 1.  Path Setting  ===
os.makedirs(output_folder_SHP, exist_ok=True)
os.makedirs(reportpath, exist_ok=True)
# === 2. ส Worker Function  ===
@dask.delayed
def original_process_logic(excel_file, target_epsg, out_folder):
    try:
        # --- [START: Logic] ---
        df = pd.read_excel(excel_file)
        if df.empty:
            return None

        if 'acq_date' in df.columns:
            df = df.drop(columns=['acq_date'])

        # Create Geodataframe from DataFrame and convert CRS
        gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))
        gdf.set_crs(epsg=4326, inplace=True)
        gdf = gdf.to_crs(epsg=target_epsg)

        # Logic for assisgning base name based on satellite and day/night
        date_str = os.path.basename(excel_file)[:8]
        date_obj = datetime.datetime.strptime(date_str, '%d%m%Y')
        doy = date_obj.timetuple().tm_yday
        year = date_obj.year
        doy_str = f"{year}_{doy:03d}"

        if "TERRA_D" in excel_file.upper():
            base_name = f"MOD11A1_LST_Day_1km_{doy_str}"
        elif "TERRA_N" in excel_file.upper():
            base_name = f"MOD11A1_LST_Night_1km_{doy_str}"
        elif "AQUA_D" in excel_file.upper():
            base_name = f"MYD11A1_LST_Day_1km_{doy_str}"
        elif "AQUA_N" in excel_file.upper():
            base_name = f"MYD11A1_LST_Night_1km_{doy_str}"
        else:
            base_name = f"UNK_LST_UNK_1km_{doy_str}"

        # Write to Disk
        output_filename = os.path.join(out_folder, f"{base_name}.shp")
        gdf.to_file(output_filename)
        # --- [END: Logic] ---

        # Returning details for report generation
        return {
            'Excel_File': os.path.basename(excel_file),
            'Shapefile': f'{base_name}.shp',
            'satellite': df['satellite'].iloc[0] if 'satellite' in df.columns else "Unknown",
            'daynight': df['daynight'].iloc[0] if 'daynight' in df.columns else "Unknown"
        }
    except Exception as e:
        print(f"❌ Error processing {excel_file}: {e}")
        return None

# === 3.  (Main) ===
if __name__ == "__main__":
    cluster = LocalCluster(n_workers=num_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit)
    client = Client(cluster)
    print(f"📈 Dashboard: {client.dashboard_link}")

    # extract all Excel files from the output directory
    excel_files = [os.path.join(Output_dir_Hotspotcluster, f) 
                   for f in os.listdir(Output_dir_Hotspotcluster) 
                   if f.endswith('.xlsx')]

    print(f"⚙️ Step: Processing {len(excel_files)} files with Dask...")

    # create a list of delayed tasks for each Excel file
    tasks = [original_process_logic(f, EPSG_target, output_folder_SHP) for f in excel_files]

    
    results = dask.compute(*tasks)

    # === 4. Summary ===
    file_details = []
    satellite_counts = Counter()

    for r in results:
        if r is not None:
            file_details.append({
                'Excel_File': r['Excel_File'],
                'Shapefile': r['Shapefile']
            })
            satellite_counts[f"{r['satellite']} ({r['daynight']})"] += 1

    print(f"\n🏁 Finished! Total files processed: {len(file_details)}")
    for key, count in satellite_counts.items():
        print(f"📊 {key}: {count} files")

    # save the report as an Excel file
    if file_details:
        details_df = pd.DataFrame(file_details)
        details_report_path = os.path.join(reportpath, 'HotspotCluster_SHP_Report.xlsx')
        details_df.to_excel(details_report_path, index=False)
        print(f"📄 Report saved at: {details_report_path}")

    client.close()
    cluster.close()

SHP TO TIFF BIT MASK

In [ ]:
%%time
import os
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
from rasterio.features import rasterize
import dask
from dask.distributed import Client, LocalCluster

# ========== 1. CONFIG & PREPARE ==========
nodata_value = -9999
os.makedirs(PTR_output, exist_ok=True)
os.makedirs(FB_output, exist_ok=True)

if __name__ == "__main__":
    cluster = LocalCluster(n_workers=num_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit)
    client = Client(cluster)
    print(f"📈 Dashboard: {client.dashboard_link}")

    # load reference raster once at the main process
    with rasterio.open(reference_raster_path) as ref_raster:
        ref_data = {
            'transform': ref_raster.transform,
            'crs': ref_raster.crs,
            'width': ref_raster.width,
            'height': ref_raster.height,
            'profile': ref_raster.profile.copy()
        }

    # ⚡ [CACHE] 
    ref_future = client.scatter(ref_data, broadcast=True)

    # extract all Shapefile paths from the output directory
    shapefile_paths = [
        os.path.join(output_folder_SHP, f)
        for f in os.listdir(output_folder_SHP)
        if f.endswith('.shp')
    ]

    # --- Worker Function  ---
    @dask.delayed
    def process_raster_dask(shp_path, ref):
        try:
            base_name = os.path.splitext(os.path.basename(shp_path))[0]
            
            # 1. load to GeoDataFrame 
            gdf = gpd.read_file(shp_path)
            if gdf.empty:
                return {"Shapefile Name": base_name, "Status": "Empty"}

            # 2. rasterize the geometries to create PTR raster
            ptr_array = rasterize(
                [(geom, 1) for geom in gdf.geometry],
                out_shape=(ref['height'], ref['width']),
                transform=ref['transform'],
                fill=nodata_value,
                dtype='float32'
            )

            
            fb_array = np.where(
                np.isnan(ptr_array) | (ptr_array == nodata_value),
                1, nodata_value
            ).astype('float32')

            # 4.prepare the final profile for writing
            final_profile = ref['profile'].copy()
            final_profile.update({
                'driver': 'GTiff',
                'height': ref['height'],
                'width': ref['width'],
                'count': 1,
                'dtype': 'float32',
                'crs': ref['crs'],
                'transform': ref['transform'],
                'nodata': nodata_value,
                'compress': 'lzw'
            })

            # 5. write to disk
            ptr_path = os.path.join(PTR_output, f"{base_name}_PTR.tif")
            fb_path = os.path.join(FB_output, f"{base_name}_FB.tif")

            with rasterio.open(ptr_path, 'w', **final_profile) as dst:
                dst.write(ptr_array, 1)
            
            with rasterio.open(fb_path, 'w', **final_profile) as dst:
                dst.write(fb_array, 1)

            return {
                "Shapefile Name": base_name,
                "Status": "Success",
                "Pixels_1": int(np.sum(fb_array == 1))
            }

        except Exception as e:
            return {"Shapefile Name": os.path.basename(shp_path), "Status": f"Error: {e}"}

    # create a list of delayed tasks for each Shapefile
    print(f"📥 waiting for {len(shapefile_paths)} files...")
    tasks = [process_raster_dask(p, ref_future) for p in shapefile_paths]

    
    print("🔥 Dask is computing...")
    results = dask.compute(*tasks)

    # ========== 3. SAVE REPORT ==========
    report_df = pd.DataFrame([r for r in results if r is not None])
    report_df.to_excel(os.path.join(reportpath, 'PTR_Dask_Report.xlsx'), index=False)
    
    client.close()
    cluster.close()
    print("🚀 All processes completed. RAM Cleared!")

Temp Correction without Hotspot

In [ ]:
%%time
import os
import shutil
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from concurrent.futures import ThreadPoolExecutor, as_completed

# === 1. CONFIG & PREPARE ===
os.makedirs(TempwithoutError, exist_ok=True)
os.makedirs(reportpath, exist_ok=True)

# function to reproject mask to match the input raster
def reproject_memory_mask(src_array, src_profile, match_src):
    dst_transform = match_src.transform
    dst_crs = match_src.crs
    dst_width = match_src.width
    dst_height = match_src.height
    dst_array = np.empty((dst_height, dst_width), dtype=np.float32)

    reproject(
        source=src_array,
        destination=dst_array,
        src_transform=src_profile['transform'],
        src_crs=src_profile['crs'],
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest
    )
    return dst_array

# === PHASE 1: LOAD ALL MASKS INTO RAM ===
mask_cache = {}
print(f"🚀 Phase 1: Loading Masks (_FB.tif) into RAM...")

if os.path.exists(FB_output):
    mask_files = [f for f in os.listdir(FB_output) if f.endswith('_FB.tif')]
    for mf in mask_files:
        mask_path = os.path.join(FB_output, mf)
        with rasterio.open(mask_path) as m_src:
            mask_cache[mf] = {
                'array': m_src.read(1),
                'profile': {
                    'transform': m_src.transform,
                    'crs': m_src.crs,
                    'height': m_src.height,
                    'width': m_src.width,
                    'nodata': m_src.nodata
                }
            }
    print(f"✔ Cached {len(mask_cache)} masks in RAM.")
else:
    print("⚠️ FB_output folder not found.")

# === PHASE 2: PROCESSING FUNCTION ===
def process_file_with_cache(input_file):
    try:
        input_path = os.path.join(Input_Raw_MODIS, input_file)
        output_path = os.path.join(TempwithoutError, input_file)
        mask_key = input_file.replace('.tif', '_FB.tif')

        with rasterio.open(input_path) as src:
            input_data = src.read(1)
            meta = src.meta.copy()
            input_nodata = src.nodata if src.nodata is not None else -9999
            meta.update(nodata=input_nodata)

            #check if the corresponding mask exists in the cache
            if mask_key not in mask_cache:
                shutil.copy(input_path, output_path)
                return {"File Name": input_file, "Status": "Mask Missing (Copied)"}

            # call the cached mask
            cached_mask = mask_cache[mask_key]
            m_array = cached_mask['array']
            m_prof = cached_mask['profile']
            m_nodata = m_prof['nodata']

            # inspect if reprojection is needed
            if (src.transform != m_prof['transform'] or 
                src.crs != m_prof['crs'] or 
                src.shape != (m_prof['height'], m_prof['width'])):
                
                mask_array_final = reproject_memory_mask(m_array, m_prof, src)
            else:
                mask_array_final = m_array

            # Apply masking logic
            # Valid areas are those where the mask is not nodata
            mask_condition = (mask_array_final != m_nodata)
            masked = np.where(mask_condition, input_data, input_nodata)

            with rasterio.open(output_path, "w", **meta) as dest:
                dest.write(masked, 1)

            return {"File Name": input_file, "Status": "Processed"}

    except Exception as e:
        return {"File Name": input_file, "Status": "Error", "Notes": str(e)}

# === PHASE 3: RUN PARALLEL ===
if __name__ == "__main__":
    tif_files = [f for f in os.listdir(Input_Raw_MODIS) if f.endswith('.tif')]
    report_data = []

    print(f"\n🚀 Phase 2: Processing {len(tif_files)} files in parallel...")
    
    # using ThreadPoolExecutor for I/O bound tasks
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(process_file_with_cache, f) for f in tif_files]
        for future in as_completed(futures):
            result = future.result()
            report_data.append(result)
            print(f"✔ {result['File Name']}: {result['Status']}")

    # Save report
    df = pd.DataFrame(report_data)
    report_name = "TemperatureWmasking_RAMMC.xlsx"
    df.to_excel(os.path.join(reportpath, report_name), index=False)
    print(f"\n✅ All Complete! Report saved to: {os.path.join(reportpath, report_name)}")

Landcover / Land-Sea Masking 

Landcover

In [ ]:
%%time
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.transform import Affine
from rasterio.mask import mask  # <--- for clipping raster with AOI
from scipy.stats import mode
import os
import glob
import geopandas as gpd
from pathlib import Path

# ───── [1] AOI PREPARATION (read, inspect the CRS) ─────
gdf_aoi = None
aoi_shapes = None  

if 'AOI_shp' in locals() or 'AOI_shp' in globals():
    aoi_path = Path(AOI_shp)
    ref_path = Path(reference_raster_path)
    
    if aoi_path.exists() and ref_path.exists():
        try:
            # 1. read AOI Shapefile
            gdf_aoi = gpd.read_file(aoi_path)
            
            # 2. read CRS from Reference Raster (LST 1km)
            with rasterio.open(ref_path) as ref_src:
                ref_crs = ref_src.crs
            
            # 3. convert AOI CRS to match Reference CRS if necessary
            if gdf_aoi.crs != ref_crs:
                print(f"🔄 CRS Mismatch: converting AOI from {gdf_aoi.crs} ⮕ {ref_crs}")
                gdf_aoi = gdf_aoi.to_crs(ref_crs)
            else:
                print(f"✅ CRS Match: AOI and Reference have matching coordinates ({ref_crs})")
            
            # 4. convert GeoDataFrame to List of Geometries (GeoJSON format) 
            aoi_shapes = [mapping(g) for g in gdf_aoi.geometry]
            print(f"📌 AOI Ready: {aoi_path.name} (ready to use for clipping)")
            
        except Exception as e:
            print(f"⚠️ Error AOI: {e}")
            aoi_shapes = None
    else:
        print(f"ℹ️ Not found AOI or Reference Raster at the specified paths")

# ───── [2] FUNCTION WITH CLIPPING ───────────────────────────────────

def downscale_snap_and_clip_majority(input_tif, reference_tif, output_tif, shapes=None):
    """
    clip and downscale the input raster to match the reference raster using majority resampling.
    """
    with rasterio.open(input_tif) as src:
        #step 1 clip the input raster if AOI shapes are provided
        if shapes is not None:
            # follow the rasterio.mask.mask() function to clip the raster
            out_image, out_transform = mask(src, shapes, crop=True)
            data = out_image[0]
            input_crs = src.crs
            input_nodata = src.nodata
            input_dtype = src.dtypes[0]
            input_transform = out_transform
        else:
            # read normally
            data = src.read(1)
            input_transform = src.transform
            input_crs = src.crs
            input_nodata = src.nodata
            input_dtype = src.dtypes[0]

    # --- Step 2: Downscale 2x2 (Majority) ---
    rows, cols = data.shape
    rows_trim = rows - rows % 2
    cols_trim = cols - cols % 2
    data_trimmed = data[:rows_trim, :cols_trim]

    reshaped = data_trimmed.reshape(rows_trim // 2, 2, cols_trim // 2, 2)
    blocks = reshaped.swapaxes(1, 2).reshape(-1, 4)
    
    # Calculate the mode (Majority)
    mode_result = mode(blocks, axis=1, nan_policy='propagate', keepdims=False)
    downscaled_data = mode_result.mode.reshape(rows_trim // 2, cols_trim // 2)

    # calculate the new transform for the downscaled raster
    new_transform = input_transform * Affine.scale(2, 2)

    # --- Step 3: Snapping to Reference Grid ---
    with rasterio.open(reference_tif) as ref:
        ref_profile = ref.profile
        ref_crs = ref.crs
        ref_transform = ref.transform
        ref_width = ref.width
        ref_height = ref.height
        ref_nodata = ref.nodata

    snapped_array = np.full((ref_height, ref_width), 
                            ref_nodata if ref_nodata is not None else 0, 
                            dtype=input_dtype)

    reproject(
        source=downscaled_data,
        destination=snapped_array,
        src_transform=new_transform,
        src_crs=input_crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        resampling=Resampling.nearest
    )

    # --- Step 4: Write Output ---
    out_profile = ref_profile.copy()
    out_profile.update({
        'driver': 'GTiff',
        'dtype': input_dtype,
        'nodata': input_nodata if input_nodata is not None else ref_nodata,
        'width': ref_width,
        'height': ref_height,
        'transform': ref_transform,
        'crs': ref_crs,
        'compress': 'lzw'
    })

    with rasterio.open(output_tif, 'w', **out_profile) as dst:
        dst.write(snapped_array.astype(input_dtype), 1)

# ───── [3] MAIN LOOP ────────────────────────────────────────────────

input_folder = LC_Folder
output_folder = outputLC_Folder
reference_path = reference_raster_path

os.makedirs(output_folder, exist_ok=True)
input_files = glob.glob(os.path.join(input_folder, "*.tif"))

print(f"🚀 start to calculation {len(input_files)} files...")

for input_tif in input_files:
    file_name = os.path.basename(input_tif)
    output_path = os.path.join(output_folder, file_name)
    
    try:
        # call function to downscale, snap, and clip the raster
        downscale_snap_and_clip_majority(input_tif, reference_path, output_path, shapes=aoi_shapes)
        print(f"✔ Done: {file_name}")
    except Exception as e:
        print(f"❌ Error {file_name}: {e}")

print("\n✅ All steps completed successfully!")

Delta Temperature 

With LC Grouping

In [ ]:
%%time
import xarray as xr
import rioxarray
import numpy as np
import os
import glob
import re
import gc
from dask.distributed import Client, LocalCluster

# ───── [1] DASK SETUP ───────────────────────────────────────────────
try:
    client.close()
    cluster.close()
except:
    pass

# configure Dask Cluster
cluster = LocalCluster(n_workers=num_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit)
client = Client(cluster)

# ───── [2] WORKER FUNCTION ──────────────────────────

def process_zonal_thermal_worker(t_path, lclw_dir, output_dir):
    """
  worker function to process a single thermal raster file, compute zonal mean based on land cover, and save the results.
    """
    try:
        # found year from the thermal raster filename
        match = re.search(r'\d{4}', os.path.basename(t_path))
        if not match: return "⏩ No year found"
        year = match.group()
        
        # find the matching land cover raster for the same year
        l_search = glob.glob(os.path.join(lclw_dir, f"*{year}*.tif"))
        if not l_search: return f"⚠️ Missing LC for {year}"
        
        l_path = l_search[0]
        base_name = os.path.basename(t_path).replace(".tif", "")
        diff_out_path = os.path.join(output_dir, f"{base_name}_difT.tif")
        mean_out_path = os.path.join(output_dir, f"{base_name}_mean.tif")

        # 1. Open Data with rioxarray (Dask-enabled)
        t_ds = rioxarray.open_rasterio(t_path, chunks=True).sel(band=1).drop_vars('band')
        l_ds = rioxarray.open_rasterio(l_path, chunks=True).sel(band=1).drop_vars('band')
        
        # 2. Reproject LC with the same grid as Thermal Raster
        l_ds = l_ds.rio.reproject_match(t_ds)
        
        # 3. calcualte Zonal Mean (only positive values)
        t_ds = t_ds.where(t_ds > 0)
        means = t_ds.groupby(l_ds).mean().compute() # compute เฉพาะค่าเฉลี่ย (ค่าจิ๋ว)
        
        # 4. zonal mean raster (broadcasting back to the grid)
        zonal_mean_raster = xr.full_like(t_ds, np.nan)
        
        conditions = []
        choices = []
        for zone_val in means.coords[means.dims[0]].values:
            val = float(means.sel({means.dims[0]: zone_val}))
            conditions.append(l_ds == zone_val)
            choices.append(val)
        
        # using dask.array.where or xarray.where
        for cond, choice in zip(conditions, choices):
            zonal_mean_raster = zonal_mean_raster.where(~cond, choice)

        # 5. calcualted Diff (Delta T)
        diff_raster = t_ds - zonal_mean_raster

        # 6. save files (Dask will pararellize the writing)
        zonal_mean_raster.rio.to_raster(mean_out_path, compress='LZW')
        diff_raster.rio.to_raster(diff_out_path, compress='LZW')

        # clear memory
        del t_ds, l_ds, means, zonal_mean_raster, diff_raster
        gc.collect()

        return f"✔ Finished: {year} ({base_name})"

    except Exception as e:
        return f"❌ Error at {t_path}: {str(e)}"

# ───── [3] MAIN EXECUTION ───────────────────────────────────────────

def solve_zonal_thermal_dask():
    temp_dir = TempwithoutError
    lclw_dir = outputLC_Folder
    output_dir = DeltaT_LC
    os.makedirs(output_dir, exist_ok=True)

    t_files = glob.glob(os.path.join(temp_dir, "*.tif"))
    print(f"🚀 เริ่มประมวลผล {len(t_files)} ไฟล์ ด้วย Dask Parallel...")

    # distriburte the workload to Dask workers
    futures = client.map(process_zonal_thermal_worker, 
                         t_files, 
                         lclw_dir=lclw_dir, 
                         output_dir=output_dir)
    
    # Conduct the computation and gather results
    results = client.gather(futures)
    
    # summary of results
    for r in results:
        print(r)

    print("\n✅ ประมวลผล Delta T เสร็จสมบูรณ์ทุกรายการ!")

if __name__ == "__main__":
    solve_zonal_thermal_dask()
    
    # close Dask client and cluster after processing
    client.close()
    cluster.close()

RETIRA Index

Original

In [ ]:
%%time
%pip install xarray rioxarray numpy dask
%pip install dask[distributed]
import os
import datetime
import numpy as np
import glob
import xarray as xr
import rioxarray
import dask
from dask.distributed import Client, LocalCluster

# ───── 1. CONFIG  ──────────────────────────────────

# create a list of (year, DOY) pairs for the specified date range
doy_year_pairs = []
curr = date_start
while curr <= date_end:
    doy_year_pairs.append((curr.year, curr.timetuple().tm_yday))
    curr += datetime.timedelta(days=1)

# Dask Settings
client = Client(n_workers=num_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit)

scenarios = [
    {'sensor': 'MOD', 'DN': 'Day'},
    {'sensor': 'MOD', 'DN': 'Night'},
    {'sensor': 'MYD', 'DN': 'Day'},
    {'sensor': 'MYD', 'DN': 'Night'}
]

# ───── 2. FUNCTIONS ────────────────────────────────────────────────

def find_file(root, sensor, dn, year, doy):
    # find the file in the root directory that matches the sensor, DN, year, and DOY
    pattern = f"*{sensor}*{dn}*{year}*{doy:03d}*.tif"
    files = glob.glob(os.path.join(root, pattern))
    return files[0] if files else None

@dask.delayed
def calculate_retira_index_delayed(baseline_paths, current_path, out_dir, tag, lbl, sensor, dn):
    try:
        prefix = f"{sensor}_{dn}_{tag}_{lbl}Y"
        os.makedirs(out_dir, exist_ok=True)
        nodata_val = -9999
        # only open the baseline datasets with rioxarray (Dask-enabled)
        baseline_ds = [rioxarray.open_rasterio(p, chunks={'x': 512, 'y': 512}, mask_and_scale=True) for p in baseline_paths]
        stack = xr.concat(baseline_ds, dim="time")
        current_da = rioxarray.open_rasterio(current_path, chunks={'x': 512, 'y': 512}, mask_and_scale=True)
        
        # calculate mean, std, and count along the time dimension
        mean_da = stack.mean(dim="time")
        std_da = stack.std(dim="time", ddof=1)
        count_da = stack.notnull().sum(dim="time")

        # Logic RETIRA
        z_da = (current_da - mean_da) / std_da
        invalid_mask = (std_da == 0) | (current_da.isnull()) | (count_da < 2)
        z_da = z_da.where(~invalid_mask, nodata_val)
        
        # save mean and std rasters
        mean_da = mean_da.where(mean_da.notnull(), nodata_val)
        std_da = std_da.where(std_da.notnull(), nodata_val)
        
        mean_da.rio.to_raster(os.path.join(out_dir, f"{prefix}_Mean.tif"))
        std_da.rio.to_raster(os.path.join(out_dir, f"{prefix}_STD.tif"))
        
        
        output_path = os.path.join(out_dir, f"{prefix}_RETIRA.tif")
        
        z_da.rio.write_nodata(nodata_val, encoded=True, inplace=True)
        z_da.rio.to_raster(output_path)
        
        return f"Done: {prefix}"
    except Exception as e:
        return f"Error: {str(e)}"

# ───── 3. MAIN LOOP  ──────────────────────────

tasks = []

# 1.Loop through each scenario (MOD/MYD and Day/Night)
for sc in scenarios:
    s, d = sc['sensor'], sc['DN']
    # assign the output directory for the current scenario
    current_out_dir = os.path.join(RETIRA_LC, f"{s}_{d}")
    
    # 2. loop through each (year, DOY) pair
    for year, doy in doy_year_pairs:
        target_file = find_file(DeltaT_LC, s, d, year, doy)
        
        if target_file:
            tag = f"{year}_{doy:03d}"
            
            # 3. loop through the baseline years (3, 5, 10)
            for lbl, n_years in {"3":3, "5":5, "10":10}.items():
                base_years = range(year - (n_years - 1), year + 1)
                valid_paths = []
                
                for by in base_years:
                    p = find_file(DeltaT_LC, s, d, by, doy)
                    if p: valid_paths.append(p)
                
                if len(valid_paths) >= 2:
                    # add the delayed task to the list
                    task = calculate_retira_index_delayed(
                        valid_paths, target_file, current_out_dir, tag, lbl, s, d
                    )
                    tasks.append(task)

print(f"gathering results: {len(tasks)} tasks submitted to Dask workers...")
print(f"starting Dask (Workers: {num_workers})...")

# excute all tasks in parallel and gather the results
results = dask.compute(*tasks)

# shown the first 5 results for quick inspection
for res in results[:5]:
    print(res)

client.close()
print(f"=== All Done ===")

Majority

LC

In [ ]:
%%time
import os
import datetime
import numpy as np
import glob
import xarray as xr
import rioxarray
from scipy.stats import mode
import warnings
import numpy as np

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice")
warnings.filterwarnings("ignore", category=FutureWarning)

# ───── 1. CONFIGURATION  ──────────────────────────
scenarios = [
    {'sensor': 'MOD', 'DN': 'Day'},
    {'sensor': 'MOD', 'DN': 'Night'},
    {'sensor': 'MYD', 'DN': 'Day'},
    {'sensor': 'MYD', 'DN': 'Night'}
]

# ───── 2. FUNCTIONS ────────────────────────────────────────────────

def find_file(root, sensor, dn, year, doy):
    pattern = f"*{sensor}*{dn}*{year}*{doy:03d}*.tif"
    files = glob.glob(os.path.join(root, pattern))
    return files[0] if files else None

def find_lc_file(root, year):
    pattern = f"*LC*{year}*.tif"
    files = glob.glob(os.path.join(root, pattern))
    return files[0] if files else None

def calculate_retira_with_debug(baseline_paths, lc_paths, current_path, current_lc_path, out_dir, tag, lbl, sensor, dn):
    try:
        prefix = f"{sensor}_{dn}_{tag}_{lbl}Y"
        os.makedirs(out_dir, exist_ok=True)
        nodata_val = -9999.0
        
        # 1. load reference raster (current year) with rioxarray
        ref_da = rioxarray.open_rasterio(current_path, mask_and_scale=True).squeeze()
        if "band" in ref_da.dims: ref_da = ref_da.drop_vars("band")

        # 2. prepare baseline and land cover datasets
        # gathering the number of years for MaxYears calculation
        max_years_count = len(baseline_paths) 
        
        baseline_ds = []
        for p in baseline_paths:
            da = rioxarray.open_rasterio(p, mask_and_scale=True).squeeze()
            if da.shape != ref_da.shape:
                da = da.rio.reproject_match(ref_da, resampling=1)
            baseline_ds.append(da)
            
        lc_ds = []
        for p in lc_paths:
            da = rioxarray.open_rasterio(p).squeeze()
            da = da.rio.reproject_match(ref_da, resampling=0)
            lc_ds.append(da)
            
        current_da = ref_da 
        current_lc = rioxarray.open_rasterio(current_lc_path).squeeze().rio.reproject_match(ref_da, resampling=0)

        # 3. Stack & Filter
        stack_dt = xr.concat(baseline_ds, dim="time")
        stack_lc = xr.concat(lc_ds, dim="time")
        
        # find the majority land cover value across the baseline years
        m = mode(stack_lc.values, axis=0, keepdims=False)
        maj_val = getattr(m, 'mode', m[0])
        majority_lc = xr.DataArray(maj_val, coords=current_da.coords, dims=current_da.dims)

        # create a mask where the land cover matches the majority and the current land cover
        valid_mask = (stack_lc == majority_lc) & (majority_lc == current_lc)
        filtered_dt = stack_dt.where(valid_mask)

        # 4. calculate statistics along the time dimension
        # Count = number of valid years (where land cover matches majority and current)
        count_da = filtered_dt.notnull().sum(dim="time")
        # MaxYears = number of baseline years (constant for all pixels)
        max_years_da = xr.full_like(ref_da, fill_value=max_years_count)
        
        mean_da = filtered_dt.mean(dim="time")
        std_da = filtered_dt.std(dim="time", ddof=1)

        # 5. calculate RETIRA index (Z-score)
        z_da = (current_da - mean_da) / std_da
        
        # condition to set RETIRA to nodata where count < 2, std == 0, or current is nodata
        final_mask = (count_da < 2) | (std_da == 0) | (current_da.isnull())
        
        # 6. prepare outputs
        outputs = {
            "RETIRA": z_da.where(~final_mask, nodata_val),
            "Count": count_da.astype(np.float32),      # จำนวนปีที่ใช้ได้จริง
            "MaxYears": max_years_da.astype(np.float32) # จำนวนปีตั้งต้น (เช่น 3, 5, 10)
        }

        # 7. save outputs to disk
        for name, da in outputs.items():
            res = da.load()
            res.rio.write_nodata(nodata_val, encoded=True, inplace=True)
            res.rio.set_crs(ref_da.rio.crs, inplace=True)
            
            out_file = os.path.join(out_dir, f"{prefix}_{name}.tif")
            res.rio.to_raster(out_file, compress='lzw', dtype=np.float32)
        
        return f"✅ Done: {prefix} (Valid: {count_da.max().values}/{max_years_count} yrs)"
        
    except Exception as e:
        return f"❌ Error on {tag}: {str(e)}"


# ───── 3. MAIN LOGIC ──────────────────────────────────────────────

# create a list of (year, DOY) pairs for the specified date range
doy_year_pairs = []
curr = date_start
while curr <= date_end:
    doy_year_pairs.append((curr.year, curr.timetuple().tm_yday))
    curr += datetime.timedelta(days=1)

print("🏃 starting Sequential (Simple Mode)...")

for sc in scenarios:
    s, d = sc['sensor'], sc['DN']
    current_out_dir = os.path.join(RETIRA_Majority_LC, f"{s}_{d}")
    
    for year, doy in doy_year_pairs:
        target_dt = find_file(DeltaT_LC, s, d, year, doy)
        target_lc = find_lc_file(outputLC_Folder, year)
        
        if target_dt and target_lc:
            tag = f"{year}_{doy:03d}"
            
            for lbl, n_years in {"3":3, "5":5, "10":10}.items():
                base_years = range(year - (n_years - 1), year + 1)
                valid_dt_paths, valid_lc_paths = [], []
                
                for by in base_years:
                    dt_p = find_file(DeltaT_LC, s, d, by, doy)
                    lc_p = find_lc_file(outputLC_Folder, by)
                    if dt_p and lc_p:
                        valid_dt_paths.append(dt_p)
                        valid_lc_paths.append(lc_p)
                
                if len(valid_dt_paths) >= 2:
                    result = calculate_retira_with_debug(
                        valid_dt_paths, valid_lc_paths, target_dt, target_lc,
                        current_out_dir, tag, lbl, s, d
                    )
                    print(result)

print("🎉 === All Done ===")

Continue / Break 

LC

In [ ]:
%%time
import os
import datetime
import numpy as np
import glob
import xarray as xr
import rioxarray
import warnings

# close warnings for cleaner output
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ───── 1. CONFIGURATION ──────────────────────────────────────────
scenarios = [
    {'sensor': 'MOD', 'DN': 'Day'},
    {'sensor': 'MOD', 'DN': 'Night'},
    {'sensor': 'MYD', 'DN': 'Day'},
    {'sensor': 'MYD', 'DN': 'Night'}
]

# ───── 2. FUNCTIONS ────────────────────────────────────────────────

def find_file(root, sensor, dn, year, doy):
    pattern = f"*{sensor}*{dn}*{year}*{doy:03d}*.tif"
    files = glob.glob(os.path.join(root, pattern))
    return files[0] if files else None

def find_lc_file(root, year):
    pattern = f"*LC*{year}*.tif"
    files = glob.glob(os.path.join(root, pattern))
    return files[0] if files else None

def calculate_retira_consecutive_debug(data_paths, lc_paths, out_dir, tag, lbl, sensor, dn):
    try:
        prefix = f"{sensor}_{dn}_{tag}_{lbl}Y"
        os.makedirs(out_dir, exist_ok=True)
        nodata_val = -9999.0
        
        # 1. load reference raster (current year) with rioxarray
        # in this case, we use the first data path as reference for reprojection
        ref_path = data_paths[0]
        ref_da = rioxarray.open_rasterio(ref_path, mask_and_scale=True).squeeze()
        if "band" in ref_da.dims: ref_da = ref_da.drop_vars("band")
        
        max_years_count = len(data_paths)

        # 2. load land cover datasets and reproject to match reference
        lc_list = []
        for p in lc_paths:
            da = rioxarray.open_rasterio(p).squeeze()
            da = da.rio.reproject_match(ref_da, resampling=0) # LC ต้อง Nearest
            lc_list.append(da)
        
        lc_stack = xr.concat(lc_list, dim="time")
        current_lc = lc_stack.isel(time=0) # ดึงปีปัจจุบันมาเป็นเกณฑ์
        
        # 3. Consecutive logic to create a valid mask
        # if the land cover matches the current year, keep it; otherwise, mask it out
        lc_match = (lc_stack == current_lc)
        valid_lc_mask = lc_match.astype(int).cumprod(dim="time").astype(bool)
        
        # 4. load Data Stack and reproject coordinates
        data_list = []
        for p in data_paths:
            da = rioxarray.open_rasterio(p, mask_and_scale=True).squeeze()
            if da.rio.width != ref_da.rio.width or da.rio.height != ref_da.rio.height:
                da = da.rio.reproject_match(ref_da, resampling=1) # อุณหภูมิใช้ Bilinear
            data_list.append(da)
            
        data_stack = xr.concat(data_list, dim="time")
        
        # Apply the valid mask to the data stack
        masked_stack = data_stack.where(valid_lc_mask)
        
        # 5. calculate statistics along the time dimension
        # count_da = number of valid years (where land cover matches current year)
        count_da = masked_stack.notnull().sum(dim="time")
        max_years_da = xr.full_like(ref_da, fill_value=max_years_count)
        
        mean_da = masked_stack.mean(dim="time")
        std_da = masked_stack.std(dim="time", ddof=1)
        
        current_da = data_stack.isel(time=0)

        # 6. calculate RETIRA index (Z-score)
        z_da = (current_da - mean_da) / std_da
        
        # if count < 2, std == 0, or current is nodata, set RETIRA to nodata
        final_mask = (count_da < 2) | (std_da == 0) | (current_da.isnull())
        
        # 7. prepare outputs
        outputs = {
            "RETIRA": z_da.where(~final_mask, nodata_val),
            "Count": count_da.astype(np.float32),      # number of valid years used
            "MaxYears": max_years_da.astype(np.float32) # number of baseline years (e.g., 3, 5, 10)
        }

        for name, da in outputs.items():
            res = da.load() # force computation to ensure data is in memory before writing
            res.rio.write_nodata(nodata_val, encoded=True, inplace=True)
            res.rio.set_crs(ref_da.rio.crs, inplace=True)
            
            out_file = os.path.join(out_dir, f"{prefix}_{name}.tif")
            res.rio.to_raster(out_file, compress='lzw', dtype=np.float32)
            
        return f"✅ Done (Consecutive): {prefix} (Continuous: {count_da.max().values}/{max_years_count})"
    
    except Exception as e:
        return f"❌ Error @ {tag}: {str(e)}"

# ───── 3. MAIN LOOP ───────────────────────────────────────────────

doy_year_pairs = []
curr = date_start
while curr <= date_end:
    doy_year_pairs.append((curr.year, curr.timetuple().tm_yday))
    curr += datetime.timedelta(days=1)

print("🚀 Starting to process Consecutive LC ...")

for sc in scenarios:
    s, d = sc['sensor'], sc['DN']
    current_out_dir = os.path.join(RETIRA_Continue_LC, f"{s}_{d}")
    
    for year, doy in doy_year_pairs:
        target_file = find_file(DeltaT_LC, s, d, year, doy)
        
        if target_file:
            tag = f"{year}_{doy:03d}"
            
            for lbl, n_years in {"3":3, "5":5, "10":10}.items():
                # sort the base years in reverse order to prioritize recent years
                base_years = sorted(range(year - (n_years - 1), year + 1), reverse=True)
                
                valid_data_paths = []
                valid_lc_paths = []
                
                for by in base_years:
                    p_data = find_file(DeltaT_LC, s, d, by, doy)
                    p_lc = find_lc_file(outputLC_Folder, by)
                    
                    if p_data and p_lc:
                        valid_data_paths.append(p_data)
                        valid_lc_paths.append(p_lc)
                    else:
                        break # if any year is missing, stop the consecutive check
                if len(valid_data_paths) >= 2:
                    result = calculate_retira_consecutive_debug(
                        valid_data_paths, valid_lc_paths, 
                        current_out_dir, tag, lbl, s, d
                    )
                    print(result)

print("🎉 จบงาน! ลองเช็คไฟล์ _Count เทียบกับ _MaxYears ดูนะนาย")

Cutoff Value with Interesting

In [ ]:
# import os
# import rioxarray
# import xarray as xr

# # --- CONFIG ---
# input_dir = RETIRA_Clip 
# output_dir = RETIRA_Final
# threshold_limit = 0
# nodata_value = -9999

# # --- PROCESS ---
# for root, dirs, files in os.walk(input_dir):
#     for file in files:
#         # check if the file is a RETIRA GeoTIFF
#         if file.lower().endswith('_retira.tif'):
            
#             input_path = os.path.join(root, file)
            
#             # manage output path
#             rel_path = os.path.relpath(root, input_dir)
#             target_folder = os.path.join(output_dir, rel_path)
#             os.makedirs(target_folder, exist_ok=True)
#             output_path = os.path.join(target_folder, file)

#             try:
#                 print(f"⌛ Processing: {file}")
                
#                 # 1. open the RETIRA raster with rioxarray (Dask-enabled)
#                 rds = rioxarray.open_rasterio(input_path, chunks=True)
                
#                 # 2. cut off values below the threshold limit (e.g., 0)
#                 # ขั้นตอนนี้จะทำให้พวกค่าขยะหรือค่าที่น้อยกว่าเกณฑ์หายไป
#                 rds_filtered = rds.where(rds < threshold_limit)
                
#                 # 3. write Header to assign NoData values
#                
#                 rds_filtered.rio.write_nodata(nodata_value, inplace=True)
                
#                 # 4. Save the filtered raster to output path with LZW compression
#                 rds_filtered.rio.to_raster(
#                     output_path,
#                     nodata=nodata_value,
#                     tiled=True,
#                     compress="lzw"
#                 )
                
#                 print(f"✅ Success: {output_path}")
                
#                 rds.close()
                
#             except Exception as e:
#                 print(f"❌ Error at {file}: {e}")

# print("\n🏁 it already")